# 3DGS Pipeline 控制中枢

**使用方法**：
1. 运行 **Cell 1（初始化）**，加载所有函数和配置
2. 按需运行各 Section 中的单元格
3. 修改参数？编辑 ，重新运行 Cell 1 即可

> 每个 Section 均可独立运行，无需依赖上方单元格的执行状态。

In [7]:
# ═══════════════════════════════════════════════════════
# ① 初始化（每次打开 Notebook 只需运行这一个 Cell）
# ═══════════════════════════════════════════════════════
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# 确保 src/ 在 Python 路径中（支持从任意目录打开 notebook）
_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
              if (p / "src" / "pipeline" / "__init__.py").exists()), Path.cwd())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.pipeline import *

cfg = load_config()   # 读取 configs/pipeline.yaml
print(f"✓ 项目根目录 : {PROJECT_ROOT}")
print(f"✓ 数据集     : {cfg['dataset']['source']}  →  {cfg['dataset']['path']}")
print(f"✓ 训练输出   : {cfg['training']['output_dir']}")
print(f"✓ 迭代次数   : {cfg['training']['iterations']}")
print(f"✓ 查看器     : {cfg['viewer']['backend']}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ 项目根目录 : /home/ansatz/github/ME6402-3D-Autonomous-Retail
✓ 数据集     : official_tandt  →  /home/ansatz/github/ME6402-3D-Autonomous-Retail/data/official/tandt_db/db/playroom
✓ 训练输出   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/playroom_5000iter
✓ 迭代次数   : 5000
✓ 查看器     : sibr


## Section 1 — 环境检查

In [4]:
# 检查 PyTorch、CUDA、COLMAP、3DGS CUDA 模块、Open3D、Docker
check_environment(cfg)


2026-03-31 20:09:18,153  INFO  环境检查完成，结果: 通过


3DGS 环境检查

PyTorch:  2.1.2
CUDA 可用: True
CUDA 版本: 11.8
  GPU 0: NVIDIA GeForce RTX 4060  (7.8 GB)
  GPU 1: NVIDIA GeForce RTX 2080 Ti  (21.7 GB)

核心依赖:
  ✓ OpenCV  4.13.0
  ✓ NumPy  1.26.4
  ✓ plyfile
  ✓ SciPy  1.15.3
  ✓ Open3D  0.19.0
  ✓ diff_gaussian_rasterization (CUDA 模块)

COLMAP:
  ✓ /usr/bin/colmap

Docker（SIBR 查看器）:
  ✓ Docker daemon 可访问

项目目录:
  PROJECT_ROOT : /home/ansatz/github/ME6402-3D-Autonomous-Retail
  GS_DIR       : /home/ansatz/github/ME6402-3D-Autonomous-Retail/third_party/gaussian-splatting  ✓
  DATA_DIR     : /home/ansatz/github/ME6402-3D-Autonomous-Retail/data  ✓
  OUTPUT_DIR   : /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs  ✓

✅ 环境检查通过


True

## Section 2 — 视频抽帧（可选）

适用场景：你有一段录制的视频，需要先抽帧再走 COLMAP 流程。

**配置方式**：在  中修改  块，将 ，
然后重新运行 **Cell 1**，再运行本 Cell。

In [ ]:
# 抽帧完成后会自动更新 cfg，指向新场景目录
# 完成后直接运行 Section 3 (COLMAP) 即可
extract_frames(cfg)


## Section 3 — COLMAP 相机标定

适用场景：自有数据（视频抽帧或自拍照片），需要从图像推导相机参数。
官方数据集（T&T、DB）已自带相机参数，**无需此步骤**。

**配置方式**：在  中将 ，
并确认  和  正确。

In [ ]:
# COLMAP 五步流程：特征提取 → 匹配 → 稀疏重建 → 畸变校正 → 内参回填
# 完成后 cfg["dataset"]["path"] 自动切换到 undistorted dense 输出
run_colmap(cfg)


## Section 4 — 3DGS 训练

关键参数（在  →  块修改）：
- ：迭代次数（300~5000 快速验证；30000 高质量）
- ：分辨率倍率（1=原始；2=1/2；RTX 4060 建议从 2 开始）
- ：输出根目录（每次训练自动创建子目录）

In [ ]:
# OOM 时自动降档重试（resolution ×1 → ×2 → ×4）
run_training(cfg)


## Section 5 — 查看训练结果

自动搜索  下最新的 ，用 Open3D 打开交互窗口。

如需查看 SIBR，在  中将 ，
重新运行 Cell 1 后再运行此 Section。

In [ ]:
# Open3D 交互查看（关闭窗口后继续）
# 也可传入指定路径：open_viewer(cfg, ply_path="outputs/xxx/point_cloud/iteration_300/point_cloud.ply")
open_viewer(cfg)


✗ 未找到 point_cloud.ply，请先完成训练（Section 4）
   搜索目录：/home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/playroom_5000iter


False

In [ ]:
# 分析训练结果：列出 PLY 文件、打印 results.json
analyze_results(cfg)


## Section 6 — SIBR Viewer（可选）

如需在 SIBR 中查看，运行此 Cell。会列出可用模型供你选择。

**前提**：已构建 Docker 镜像（见 [info] Building image: sibr-builder:ubuntu22.04-cuda11.8
Sending build context to Docker daemon  12.01GB

Step 1/4 : FROM nvidia/cuda:11.8.0-devel-ubuntu22.04
 ---> 6f9cc9f1ba9e
Step 2/4 : ENV DEBIAN_FRONTEND=noninteractive
 ---> Using cache
 ---> db877d8112e1
Step 3/4 : RUN apt-get update && apt-get install -y --no-install-recommends     build-essential     cmake     ninja-build     git     pkg-config     libglew-dev     libassimp-dev     libboost-all-dev     libgtk-3-dev     libopencv-dev     libglfw3-dev     libavdevice-dev     libavcodec-dev     libavformat-dev     libswscale-dev     libeigen3-dev     libxxf86vm-dev     libembree-dev     libgl1-mesa-dev     libglu1-mesa-dev     libx11-dev     libxext-dev     libxrender-dev     libxrandr-dev     libxinerama-dev     libxcursor-dev     ca-certificates     && rm -rf /var/lib/apt/lists/*
 ---> Using cache
 ---> feb68a7c28aa
Step 4/4 : WORKDIR /workspace
 ---> Using cache
 ---> 8fdbd6c2ecd5
Successfully built 8fdbd6c2ecd5
Successfully tagged sibr-builder:ubuntu22.04-cuda11.8
[done] Image built: sibr-builder:ubuntu22.04-cuda11.8）

In [14]:
# 交互式选择模型并在 Docker 内启动 SIBR
launch_sibr(cfg)



可用模型（按最新迭代降序）:
  [1] 7dfdb283-b  (iter=30000)
  [2] 3dgs_tandt_30000iter  (iter=30000)
  [3] 3dgs_tandt_5000iter  (iter=5000)
  [4] 3dgs_custom_scene_01_5000iter  (iter=5000)
  [5] 3dgs_demo_300iter  (iter=300)
  [6] smoke_truck_10iter  (iter=10)
  [7] 3dgs_demo  (iter=?)

🖼️  启动 SIBR Viewer ...
   bash /home/ansatz/github/ME6402-3D-Autonomous-Retail/scripts/reconstruction/run_sibr_in_docker.sh sibr-builder:ubuntu22.04-cuda11.8 /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/Teamate_output_unzipped/content/gaussian-splatting/output/7dfdb283-b
   提示：关闭 SIBR 窗口后，该单元继续运行。
[info] Launching SIBR viewer in container...

== CUDA ==

CUDA Version 11.8.0

Container image Copyright (c) 2016-2023, NVIDIA CORPORATION & AFFILIATES. All rights reserved.

This container image and its contents are governed by the NVIDIA Deep Learning Container License.
By pulling and using the container, you accept the terms and conditions of this license:
https://developer.nvidia.com/ngc/nvidia-deep-learning

[SIBR] ##  ERROR  ##:	FILE /workspace/third_party/gaussian-splatting/SIBR_viewers/src/core/scene/ParseData.cpp
			LINE 560, FUNC getParsedData
			Cannot determine type of dataset at //content/gaussian-splatting/data


[SIBR] --  INFOS  --:	Initializing Raycaster
[SIBR] --  INFOS  --:	Interactive camera using (0.009,1100) near/far planes.
Switched to trackball mode.


2026-03-31 20:42:17,425  INFO  SIBR 查看完成: /home/ansatz/github/ME6402-3D-Autonomous-Retail/outputs/Teamate_output_unzipped/content/gaussian-splatting/output/7dfdb283-b，ok=True


[SIBR] --  INFOS  --:	Deinitialization of GLFW
[done] SIBR viewer exited.
✓ SIBR 正常退出


True

## Section 7 — 一键完整流程

参数调好后，一口气执行：COLMAP（可选）→ 训练 → 查看结果。

In [ ]:
# 一键流程（是否跑 COLMAP 取决于 cfg["dataset"]["use_colmap"]）
run_pipeline(cfg)


## Section 8 — 日志

In [ ]:
import subprocess
from src.pipeline import LOG_DIR

logs = sorted(LOG_DIR.glob("pipeline_*.log"), reverse=True)
if logs:
    latest = logs[0]
    print(f"最新日志：{latest}
" + "-"*50)
    # 打印最后 30 行
    lines = latest.read_text(encoding="utf-8").splitlines()
    print("
".join(lines[-30:]))
else:
    print(f"日志目录：{LOG_DIR}（暂无日志）")
